# Prefect + Dask

This is a tutorial to use the Prefect and Dask clusters together from Jupyter.

NOTE: we can use the EOPF Dask cluster from Prefect, but not the staging cluster because of dependency conflicts.

See the associated Python module: [my_prefect_and_dask.py](./my_prefect_and_dask.py)

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['PREFECT_PUBLIC']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

await init_prefect_blocks()
init_demo()
init_dask_cluster_eopf(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import *

Auxip service: http://rs-server-adgs:8000
CADIP service: http://rs-server-cadip:8000
Catalog service: http://rs-server-catalog:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...


KeyError: 'LOCAL_GATEWAY_USERNAME'

In [ ]:
%%bash
prefect block ls
echo -e "\nNOTE: you can see the block details and credentials by running e.g.: prefect block inspect remote-file-system/s3"

In [ ]:
# Other imports
import getpass
import os
import prefect
from resources import prefect_utils
from distributed.client import Future

# Data to test the example flow
my_data = {"start": "2000", "end": "2001", "freq": "2w"}

# Data as a command-line string in the json format
my_data_str = json.dumps(my_data)
my_data_str = my_data_str.replace('"', r'\"')

In [ ]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf

# Save cluster info to be read by our flow
os.environ["DASK_GATEWAY_ADDRESS"] = dask_gateway.address
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Set environment variables for the client and dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"
dask_client.run(set_dask_env)
os.environ["HELLO_FROM"] = "client"

# 1. Call Prefect flow from Python code

In [ ]:
# Call flow and print results
from my_prefect_and_dask import my_flow
result = my_flow(**my_data)
display(result.head()) # this one doesn't work when deploying prefect in the next section, I don't know why

<div class="alert alert-info" role="alert">
Notes:

  1. In the Prefect dashboard (see link above), find your run, check its graph and logs.
  1. Check in the dashboard and above logs above that when we call the flow from Python code, no Prefect workers are involved:
      1. Your module main code (outside functions) and flow are run only by client.
      1. Your module tasks are run only by the Dask workers.
  1. If you want to check the IP adresses:
      1. On kubernetes, you can run the `kubectl describe` command to check a pod IP address.
      1. In local mode, use: `docker inspect <container_id> | grep IPAddress`

# 2. Deploy Prefect flow

See the full yaml file: [deploy-prefect-dask.yaml](./deploy-prefect-dask.yaml)

First we deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_folder = f"users/{getpass.getuser()}" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code and wheel to: '{PREFECT_BLOCK_S3.basepath}/{s3_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_folder}/resources")

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./deploy-prefect-dask.yaml"

In [ ]:
deploy_name = "my-flow/tuto-prefect-dask"
await prefect_utils.wait_for_deployment(deploy_name)

In [ ]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2"

## 3. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway_eopf)

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.